## Setup común para los tests del generador

Este bloque importa `generate_case.py`, fija el orden de las 9 configuraciones y define
helpers pequeños para liberar memoria entre casos.

Si `generate_case.py` no está en el mismo directorio del notebook, ajusta `MODULE_DIR`.

In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path("../../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [2]:
from __future__ import annotations

from pathlib import Path
import sys
import gc as py_gc
import json
import tempfile

import pandas as pd
from pandas.testing import assert_frame_equal


from experiments.persistence_formats.generate_case import (
    PREDEFINED_CONFIGS,
    DEFAULT_SEED,
    compute_expected_fingerprint,
    generate_experiment_case,
    get_predefined_config,
)

CONFIG_IDS = ["C1", "C2", "C3", "C4", "C5", "C6", "C7", "C8", "C9"]


def _cleanup_case(case):
    del case
    py_gc.collect()


def _print_ok(msg: str) -> None:
    print(f"Ok: {msg}")

## Test 1 - Smoke test completo de las 9 configuraciones

Objetivo:
- generar las 9 configuraciones sin error,
- verificar que respetan `(C, R, k)`,
- verificar que producen el número correcto de columnas,
- verificar la cardinalidad correcta de `exp_cat_a_k` y `exp_cat_b_k`,
- verificar que el fingerprint esperado queda construido con la estructura mínima esperada.

In [3]:
results = []

for config_id in CONFIG_IDS:
    config = get_predefined_config(config_id)
    case = generate_experiment_case(config, seed=DEFAULT_SEED)

    df = case.trips.data
    fp = case.expected_fingerprint
    manifest = case.manifest

    # 1) shape y cantidad de columnas
    assert df.shape == (config.n_rows, config.n_cols), (
        f"{config_id}: shape incorrecto. "
        f"esperado={(config.n_rows, config.n_cols)}, obtenido={df.shape}"
    )
    assert len(df.columns) == config.n_cols, (
        f"{config_id}: número de columnas incorrecto. "
        f"esperado={config.n_cols}, obtenido={len(df.columns)}"
    )

    # 2) manifest coherente con config
    assert manifest["config_id"] == config.config_id
    assert manifest["n_rows"] == config.n_rows
    assert manifest["n_cols"] == config.n_cols
    assert manifest["k"] == config.k

    # 3) fingerprint con estructura mínima esperada
    assert fp["config_id"] == config.config_id
    assert fp["seed"] == DEFAULT_SEED
    assert fp["n_rows"] == config.n_rows
    assert fp["n_cols"] == config.n_cols
    assert fp["k"] == config.k
    assert isinstance(fp["columns_order"], list)
    assert isinstance(fp["dtypes_expected"], dict)
    assert isinstance(fp["column_fingerprints"], dict)
    assert isinstance(fp["dataset_fingerprint"], str)
    assert len(fp["columns_order"]) == config.n_cols
    assert len(fp["column_fingerprints"]) == config.n_cols

    # 4) cardinalidad observada de los 2 campos controlados
    card_a = df["exp_cat_a_k"].nunique(dropna=True)
    card_b = df["exp_cat_b_k"].nunique(dropna=True)
    assert card_a == config.k, (
        f"{config_id}: exp_cat_a_k cardinalidad incorrecta. "
        f"esperado={config.k}, obtenido={card_a}"
    )
    assert card_b == config.k, (
        f"{config_id}: exp_cat_b_k cardinalidad incorrecta. "
        f"esperado={config.k}, obtenido={card_b}"
    )

    # 5) cardinalidad de categorías declaradas
    assert len(df["exp_cat_a_k"].cat.categories) == config.k, (
        f"{config_id}: exp_cat_a_k categories incorrectas"
    )
    assert len(df["exp_cat_b_k"].cat.categories) == config.k, (
        f"{config_id}: exp_cat_b_k categories incorrectas"
    )

    results.append(
        {
            "config_id": config_id,
            "shape": df.shape,
            "card_a": card_a,
            "card_b": card_b,
            "dataset_fingerprint_prefix": fp["dataset_fingerprint"][:12],
        }
    )

    _cleanup_case(case)

results_df = pd.DataFrame(results)
display(results_df)
_print_ok("Test 1 superado: generación, shape, columnas, cardinalidades y fingerprint mínimo.")

,config_id,shape,card_a,card_b,dataset_fingerprint_prefix
0,C1,"(200000, 36)",10,10,3f9af838fb90
1,C2,"(200000, 156)",10,10,62c82cac6a35
2,C3,"(200000, 256)",10,10,238689ae5886
3,C4,"(20000, 156)",10,10,178d9db5a094
4,C5,"(1000000, 156)",10,10,fb3689520649
5,C6,"(200000, 156)",1000,1000,2f733085a0fe
6,C7,"(200000, 156)",10000,10000,16caee5ebfb3
7,C8,"(1000000, 256)",10,10,ec28f3789f43
8,C9,"(1000000, 156)",10000,10000,803537920bb2


Ok: Test 1 superado: generación, shape, columnas, cardinalidades y fingerprint mínimo.


## Test 2 - Verificación del contrato del TripSchema y del schema_effective

Objetivo:
- comprobar que el `TripSchema` contiene los campos base esperados,
- comprobar que `schema_effective` refleja el DataFrame generado,
- comprobar que los campos categóricos controlados y fijos quedaron tipados como `categorical`.

In [4]:
BASE_REQUIRED_FIELDS = [
    "movement_id",
    "user_id",
    "origin_longitude",
    "origin_latitude",
    "destination_longitude",
    "destination_latitude",
    "origin_h3_index",
    "destination_h3_index",
    "origin_time_utc",
    "destination_time_utc",
    "trip_id",
    "movement_seq",
]

EXPECTED_FIXED_CATEGORICALS = ["mode", "purpose", "day_type", "time_period"]
EXPECTED_CONTROLLED_CATEGORICALS = ["exp_cat_a_k", "exp_cat_b_k"]

for config_id in CONFIG_IDS:
    config = get_predefined_config(config_id)
    case = generate_experiment_case(config, seed=DEFAULT_SEED)

    df = case.trips.data
    schema = case.schema
    schema_eff = case.trips.schema_effective

    # 1) required del schema
    for field in BASE_REQUIRED_FIELDS:
        assert field in schema.required, f"{config_id}: falta required {field}"
        assert field in schema.fields, f"{config_id}: falta FieldSpec para {field}"

    # 2) coherencia entre schema_effective y dataframe
    assert schema_eff.fields_effective == list(df.columns), (
        f"{config_id}: fields_effective no coincide con columnas del DataFrame"
    )

    for col in df.columns:
        assert col in schema_eff.dtype_effective, (
            f"{config_id}: falta dtype_effective para columna {col}"
        )

    # 3) tipado lógico de categóricos
    for field in EXPECTED_FIXED_CATEGORICALS + EXPECTED_CONTROLLED_CATEGORICALS:
        assert schema.fields[field].dtype == "categorical", (
            f"{config_id}: {field} no quedó como categorical en TripSchema"
        )
        assert field in schema_eff.domains_effective, (
            f"{config_id}: {field} no quedó en domains_effective"
        )

    _cleanup_case(case)

_print_ok("Test 2 superado: TripSchema y schema_effective coherentes.")

Ok: Test 2 superado: TripSchema y schema_effective coherentes.


## Test 3 - Construcción y persistencia del fingerprint esperado

Objetivo:
- comprobar que el fingerprint esperado se puede recomputar,
- comprobar que `persist_fingerprint_dir` escribe un JSON por configuración,
- comprobar que el JSON persistido coincide exactamente con el fingerprint retornado.

In [5]:
with tempfile.TemporaryDirectory() as tmpdir:
    tmpdir_path = Path(tmpdir)

    persisted = []

    for config_id in CONFIG_IDS:
        config = get_predefined_config(config_id)
        case = generate_experiment_case(
            config,
            seed=DEFAULT_SEED,
            persist_fingerprint_dir=tmpdir_path,
        )

        print("\nDataframe generado y fingerprint generado")
        display(case.trips.data)
        display(case.expected_fingerprint)

        # 1) archivo generado
        assert case.fingerprint_path is not None, f"{config_id}: fingerprint_path es None"
        assert case.fingerprint_path.exists(), f"{config_id}: no existe el JSON persistido"

        # 2) contenido persistido = fingerprint retornado
        persisted_payload = json.loads(case.fingerprint_path.read_text(encoding="utf-8"))
        assert persisted_payload == case.expected_fingerprint, (
            f"{config_id}: JSON persistido distinto del fingerprint retornado"
        )

        # 3) recomputación directa del fingerprint sobre el DF
        recomputed = compute_expected_fingerprint(
            case.trips.data,
            config=config,
            seed=DEFAULT_SEED,
        )
        assert recomputed == case.expected_fingerprint, (
            f"{config_id}: fingerprint recomputado distinto del original"
        )

        persisted.append(
            {
                "config_id": config_id,
                "path": str(case.fingerprint_path),
                "dataset_fingerprint_prefix": case.expected_fingerprint["dataset_fingerprint"][:12],
            }
        )

        _cleanup_case(case)

persisted_df = pd.DataFrame(persisted)
display(persisted_df)
_print_ok("Test 3 superado: fingerprint construido, recomputable y persistible.")


Dataframe generado y fingerprint generado


,movement_id,user_id,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_h3_index,destination_h3_index,origin_time_utc,destination_time_utc,trip_id,movement_seq,trip_weight,mode,purpose,day_type,time_period,exp_cat_a_k,exp_cat_b_k,extra_float_001,extra_float_002,extra_float_003,extra_float_004,extra_float_005,extra_float_006,extra_float_007,extra_float_008,extra_float_009,extra_float_010,extra_int_011,extra_int_012,extra_int_013,extra_int_014,extra_int_015,extra_bool_016,extra_bool_017
0,m000000000,u0000000,-70.548698,-33.442817,-70.644262,-33.457560,88b2c50b5dfffff,88b2c554ebfffff,2024-01-01 00:00:00,2024-01-01 00:57:00,t000000000,0,1.789122,walk,work,weekday,early_morning,catA_000000000000000,catB_000000000000000,-6.569583,-2.538557,-5.300779,-2.618196,0.165055,-0.539054,0.040573,1.830845,3.453282,2.957338,19628,16427,7814,36029,37737,False,False
1,m000000001,u0000001,-70.626188,-33.459854,-70.643105,-33.462717,88b2c554d5fffff,88b2c5548dfffff,2024-01-01 00:05:00,2024-01-01 01:25:00,t000000001,0,0.659620,bus,health,weekend,night,catA_000000000000001,catB_000000000000007,-6.470406,-3.998699,-2.448689,-2.230184,-3.373983,1.461114,-0.048794,3.204721,5.840623,3.194168,3269,63741,55384,5475,33704,True,True
2,m000000002,u0000002,-70.477309,-33.358739,-70.441733,-33.359257,88b2c51aa1fffff,88b2c51a95fffff,2024-01-01 00:10:00,2024-01-01 01:20:00,t000000002,0,0.925248,metro,home,weekday,evening_peak,catA_000000000000002,catB_000000000000004,-4.145405,-5.252392,-3.630189,0.038562,-0.508616,0.087300,0.452722,1.365653,3.681952,7.214400,12815,11640,14707,7573,9921,False,True
3,m000000003,u0000003,-70.647832,-33.469403,-70.584782,-33.500681,88b2c55485fffff,88b2c50955fffff,2024-01-01 00:15:00,2024-01-01 01:06:00,t000000003,0,2.569640,car,study,weekend,afternoon,catA_000000000000003,catB_000000000000001,-3.726217,-3.623820,-3.500259,-2.229123,0.146245,-0.640747,1.658730,1.386366,4.006337,6.662722,22405,65216,72388,51956,92437,False,True
4,m000000004,u0000004,-70.751084,-33.399419,-70.662427,-33.386542,88b2c55725fffff,88b2c5562bfffff,2024-01-01 00:20:00,2024-01-01 01:10:00,t000000004,0,2.582844,bike,care,weekday,midday,catA_000000000000004,catB_000000000000008,-4.806444,-3.746562,-2.017984,-0.307787,-1.548323,-0.894276,-0.435397,1.542092,4.076447,3.430614,13287,18953,53921,43757,81925,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,m000199995,u0039995,-70.707915,-33.439420,-70.651041,-33.430997,88b2c55425fffff,88b2c55419fffff,2024-11-25 10:15:00,2024-11-25 10:50:00,t000199995,0,2.730107,car,study,weekend,afternoon,catA_000000000000005,catB_000000000000005,-3.023697,-4.223142,-2.124107,-2.011173,0.643956,0.216391,1.039833,-0.287892,3.121410,4.698740,15577,43084,98191,82270,98409,False,True
199996,m000199996,u0039996,-70.711593,-33.325395,-70.732408,-33.281354,88b2c5cdedfffff,88b2c5cd45fffff,2024-11-25 10:20:00,2024-11-25 10:48:00,t000199996,0,2.800921,bike,care,weekday,midday,catA_000000000000006,catB_000000000000002,-3.443176,-3.968946,-3.154173,-2.754786,-1.856022,0.784983,2.254236,1.281951,4.027695,4.072890,81147,83999,84869,61533,27130,False,False
199997,m000199997,u0039997,-70.555897,-33.328866,-70.568579,-33.280331,88b2c51b39fffff,88b2c5ce99fffff,2024-11-25 10:25:00,2024-11-25 11:30:00,t000199997,0,1.745985,scooter,other,weekend,morning_peak,catA_000000000000007,catB_000000000000009,-4.225336,-4.015649,-5.043242,-2.954792,-1.141486,-0.464817,0.392650,2.489522,1.767515,3.716872,23754,55198,72942,72462,79747,True,False
199998,m000199998,u0039998,-70.695768,-33.378685,-70.677198,-33.390750,88b2c55751fffff,88b2c55621fffff,2024-11-25 10:30:00,2024-11-25 10:39:00,t000199998,0,2.244973,walk,shopping,weekday,early_morning,catA_000000000000008,catB_000000000000006,-6.227163,-3.873233,-2.508353,-1.444413,-1.791821,0.288404,-0.407458,2.471392,3.094915,4.539600,75588,4444,40666,27173,42147,True,False


{'config_id': 'C1',
 'seed': 20260415,
 'n_rows': 200000,
 'n_cols': 36,
 'k': 10,
 'columns_order': ['movement_id',
  'user_id',
  'origin_longitude',
  'origin_latitude',
  'destination_longitude',
  'destination_latitude',
  'origin_h3_index',
  'destination_h3_index',
  'origin_time_utc',
  'destination_time_utc',
  'trip_id',
  'movement_seq',
  'trip_weight',
  'mode',
  'purpose',
  'day_type',
  'time_period',
  'exp_cat_a_k',
  'exp_cat_b_k',
  'extra_float_001',
  'extra_float_002',
  'extra_float_003',
  'extra_float_004',
  'extra_float_005',
  'extra_float_006',
  'extra_float_007',
  'extra_float_008',
  'extra_float_009',
  'extra_float_010',
  'extra_int_011',
  'extra_int_012',
  'extra_int_013',
  'extra_int_014',
  'extra_int_015',
  'extra_bool_016',
  'extra_bool_017'],
 'dtypes_expected': {'movement_id': 'string',
  'user_id': 'string',
  'origin_longitude': 'float64',
  'origin_latitude': 'float64',
  'destination_longitude': 'float64',
  'destination_latitude': 


Dataframe generado y fingerprint generado


,movement_id,user_id,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_h3_index,destination_h3_index,origin_time_utc,destination_time_utc,trip_id,movement_seq,trip_weight,mode,purpose,day_type,time_period,exp_cat_a_k,exp_cat_b_k,extra_float_001,extra_float_002,extra_float_003,extra_float_004,extra_float_005,extra_float_006,extra_float_007,extra_float_008,extra_float_009,extra_float_010,extra_int_011,extra_int_012,extra_int_013,extra_int_014,extra_int_015,extra_bool_016,extra_bool_017,extra_bool_018,extra_string_019,extra_datetime_020,extra_float_021,...,extra_bool_098,extra_string_099,extra_datetime_100,extra_float_101,extra_float_102,extra_float_103,extra_float_104,extra_float_105,extra_float_106,extra_float_107,extra_float_108,extra_float_109,extra_float_110,extra_int_111,extra_int_112,extra_int_113,extra_int_114,extra_int_115,extra_bool_116,extra_bool_117,extra_bool_118,extra_string_119,extra_datetime_120,extra_float_121,extra_float_122,extra_float_123,extra_float_124,extra_float_125,extra_float_126,extra_float_127,extra_float_128,extra_float_129,extra_float_130,extra_int_131,extra_int_132,extra_int_133,extra_int_134,extra_int_135,extra_bool_136,extra_bool_137
0,m000000000,u0000000,-70.548698,-33.442817,-70.644262,-33.457560,88b2c50b5dfffff,88b2c554ebfffff,2024-01-01 00:00:00,2024-01-01 00:57:00,t000000000,0,1.789122,walk,work,weekday,early_morning,catA_000000000000000,catB_000000000000000,-6.569583,-2.538557,-5.300779,-2.618196,0.165055,-0.539054,0.040573,1.830845,3.453282,2.957338,19628,16427,7814,36029,37737,False,False,False,estr_019_00,2024-01-01 19:00:00,6.170077,...,True,estr_099_00,2024-01-05 03:00:00,-1.959527,-2.857349,-3.795181,0.115331,0.249721,0.698366,2.472459,3.660157,3.977093,4.348375,72644,19622,53999,87006,86411,True,True,True,estr_119_00,2024-01-05 23:00:00,6.995404,-3.499496,-2.196522,-2.357504,-2.948996,-5.133727,-2.508230,1.759061,0.908775,2.289840,27794,74798,82444,40493,87041,False,False
1,m000000001,u0000001,-70.626188,-33.459854,-70.643105,-33.462717,88b2c554d5fffff,88b2c5548dfffff,2024-01-01 00:05:00,2024-01-01 01:25:00,t000000001,0,0.659620,bus,health,weekend,night,catA_000000000000001,catB_000000000000007,-6.470406,-3.998699,-2.448689,-2.230184,-3.373983,1.461114,-0.048794,3.204721,5.840623,3.194168,3269,63741,55384,5475,33704,True,True,True,estr_019_01,2024-01-01 19:07:00,4.556074,...,False,estr_099_01,2024-01-05 03:09:00,-1.563570,-0.650349,-3.033507,-2.435132,-1.950747,1.233817,0.603542,4.370375,6.588776,2.008350,52048,73385,59319,44013,98288,True,True,True,estr_119_01,2024-01-05 23:03:00,3.038314,-3.272005,-3.593040,-4.058341,-1.868071,-1.734937,0.210827,1.033136,1.153492,1.524552,30684,66731,89439,37633,9553,True,True
2,m000000002,u0000002,-70.477309,-33.358739,-70.441733,-33.359257,88b2c51aa1fffff,88b2c51a95fffff,2024-01-01 00:10:00,2024-01-01 01:20:00,t000000002,0,0.925248,metro,home,weekday,evening_peak,catA_000000000000002,catB_000000000000004,-4.145405,-5.252392,-3.630189,0.038562,-0.508616,0.087300,0.452722,1.365653,3.681952,7.214400,12815,11640,14707,7573,9921,False,True,False,estr_019_02,2024-01-01 19:14:00,3.803799,...,False,estr_099_02,2024-01-05 03:18:00,-4.731805,-3.762322,-2.144255,0.348413,-0.437894,-0.606230,0.548473,2.764573,7.073274,8.671947,23147,41989,79498,105,60846,True,False,True,estr_119_02,2024-01-05 23:06:00,4.160029,-4.994949,-4.881060,-2.727592,-2.344853,-1.929595,0.590174,1.633487,-0.049978,2.114967,52274,80266,42015,15173,15240,False,True
3,m000000003,u0000003,-70.647832,-33.469403,-70.584782,-33.500681,88b2c55485fffff,88b2c50955fffff,2024-01-01 00:15:00,2024-01-01 01:06:00,t000000003,0,2.569640,car,study,weekend,afternoon,catA_000000000000003,catB_000000000000001,-3.726217,-3.623820,-3.500259,-2.229123,0.146245,-0.640747,1.658730,1.386366,4.006337,6.662722,22405,65216,72388,51956,92437,False,True,False,estr_019_03,2024-01-01 19:21:00,2.758671,...,True,estr_099_03,2024-01-05 03:27:00,-4.647172,-2.553841,-4.534768,-2.305204,-0.330

...


Dataframe generado y fingerprint generado


,movement_id,user_id,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_h3_index,destination_h3_index,origin_time_utc,destination_time_utc,trip_id,movement_seq,trip_weight,mode,purpose,day_type,time_period,exp_cat_a_k,exp_cat_b_k,extra_float_001,extra_float_002,extra_float_003,extra_float_004,extra_float_005,extra_float_006,extra_float_007,extra_float_008,extra_float_009,extra_float_010,extra_int_011,extra_int_012,extra_int_013,extra_int_014,extra_int_015,extra_bool_016,extra_bool_017,extra_bool_018,extra_string_019,extra_datetime_020,extra_float_021,...,extra_bool_098,extra_string_099,extra_datetime_100,extra_float_101,extra_float_102,extra_float_103,extra_float_104,extra_float_105,extra_float_106,extra_float_107,extra_float_108,extra_float_109,extra_float_110,extra_int_111,extra_int_112,extra_int_113,extra_int_114,extra_int_115,extra_bool_116,extra_bool_117,extra_bool_118,extra_string_119,extra_datetime_120,extra_float_121,extra_float_122,extra_float_123,extra_float_124,extra_float_125,extra_float_126,extra_float_127,extra_float_128,extra_float_129,extra_float_130,extra_int_131,extra_int_132,extra_int_133,extra_int_134,extra_int_135,extra_bool_136,extra_bool_137
0,m000000000,u0000000,-70.548698,-33.375225,-70.532110,-33.363752,88b2c5191bfffff,88b2c5182bfffff,2024-01-01 00:00:00,2024-01-01 01:17:00,t000000000,0,3.022054,walk,work,weekday,early_morning,catA_000000000000000,catB_000000000000000,-5.225485,-2.383964,-5.797504,-0.252164,-1.931505,1.536431,0.139336,0.324634,1.201582,4.567665,41563,30047,73834,14836,92361,True,False,False,estr_019_00,2024-01-01 19:00:00,3.753587,...,True,estr_099_00,2024-01-05 03:00:00,-4.543994,-1.709293,-2.067439,2.487197,1.966566,0.699724,1.001896,4.520316,4.192260,3.887886,5702,93549,51815,41167,56982,True,True,False,estr_119_00,2024-01-05 23:00:00,3.926505,-2.858655,-2.517974,-3.232413,-0.639838,-1.024960,0.767472,1.187960,2.834184,3.865673,23927,67535,75952,22602,15202,False,True
1,m000000001,u0000001,-70.626188,-33.497693,-70.660142,-33.524017,88b2c50965fffff,88b2c54603fffff,2024-01-01 00:05:00,2024-01-01 00:43:00,t000000001,0,0.516663,bus,health,weekend,night,catA_000000000000001,catB_000000000000007,-6.325211,-4.300017,-3.620786,-2.541908,-2.153775,1.261049,1.560304,1.431242,2.455902,3.787120,1265,99518,45754,80779,90642,False,True,False,estr_019_01,2024-01-01 19:07:00,3.631266,...,True,estr_099_01,2024-01-05 03:09:00,-7.037268,-3.293097,-2.124763,-2.659942,-0.368163,1.007228,2.378378,3.441283,1.191240,2.767853,63743,97757,75002,14876,11852,False,False,False,estr_119_01,2024-01-05 23:03:00,4.953009,-3.801310,-5.124813,-3.355264,-2.755631,-1.048584,-0.899560,0.507560,2.743222,3.713245,2859,44179,97468,12203,32634,True,True
2,m000000002,u0000002,-70.477309,-33.607227,-70.490471,-33.659227,88b2c572c1fffff,88b2c570ddfffff,2024-01-01 00:10:00,2024-01-01 01:10:00,t000000002,0,1.788918,metro,home,weekday,evening_peak,catA_000000000000002,catB_000000000000014,-5.073300,-2.314768,-4.086955,-1.793539,-0.620870,-1.664360,0.057233,1.211968,3.459610,3.420410,93632,51727,55810,37671,74920,True,True,False,estr_019_02,2024-01-01 19:14:00,4.426758,...,False,estr_099_02,2024-01-05 03:18:00,-4.680357,-0.765601,-3.874007,0.373408,1.281325,1.968661,-0.517163,3.875226,2.740986,3.932971,82377,40944,93707,91132,10105,False,False,False,estr_119_02,2024-01-05 23:06:00,7.468490,-4.684500,-3.222500,-1.620252,-4.420791,-0.971393,1.612893,-0.620729,2.286368,2.203187,82935,88878,41715,32571,5621,True,False
3,m000000003,u0000003,-70.647832,-33.617174,-70.669247,-33.633709,88b2c544b9fffff,88b2c57a49fffff,2024-01-01 00:15:00,2024-01-01 00:35:00,t000000003,0,2.091992,car,study,weekend,afternoon,catA_000000000000003,catB_000000000000021,-4.847829,-2.228616,-3.628575,-1.665521,-0.509252,1.790039,2.596956,4.200410,2.040542,4.096063,49366,51627,77317,33849,55439,False,True,True,estr_019_03,2024-01-01 19:21:00,6.238249,...,True,estr_099_03,2024-01-05 03:27:00,-3.624232,-2.113253,-3.013742,-2.518076

{'config_id': 'C9',
 'seed': 20260415,
 'n_rows': 1000000,
 'n_cols': 156,
 'k': 10000,
 'columns_order': ['movement_id',
  'user_id',
  'origin_longitude',
  'origin_latitude',
  'destination_longitude',
  'destination_latitude',
  'origin_h3_index',
  'destination_h3_index',
  'origin_time_utc',
  'destination_time_utc',
  'trip_id',
  'movement_seq',
  'trip_weight',
  'mode',
  'purpose',
  'day_type',
  'time_period',
  'exp_cat_a_k',
  'exp_cat_b_k',
  'extra_float_001',
  'extra_float_002',
  'extra_float_003',
  'extra_float_004',
  'extra_float_005',
  'extra_float_006',
  'extra_float_007',
  'extra_float_008',
  'extra_float_009',
  'extra_float_010',
  'extra_int_011',
  'extra_int_012',
  'extra_int_013',
  'extra_int_014',
  'extra_int_015',
  'extra_bool_016',
  'extra_bool_017',
  'extra_bool_018',
  'extra_string_019',
  'extra_datetime_020',
  'extra_float_021',
  'extra_float_022',
  'extra_float_023',
  'extra_float_024',
  'extra_float_025',
  'extra_float_026',
  

,config_id,path,dataset_fingerprint_prefix
0,C1,C:\Users\sebam\AppData\Local\Temp\tmpardfzz93\...,3f9af838fb90
1,C2,C:\Users\sebam\AppData\Local\Temp\tmpardfzz93\...,62c82cac6a35
2,C3,C:\Users\sebam\AppData\Local\Temp\tmpardfzz93\...,238689ae5886
3,C4,C:\Users\sebam\AppData\Local\Temp\tmpardfzz93\...,178d9db5a094
4,C5,C:\Users\sebam\AppData\Local\Temp\tmpardfzz93\...,fb3689520649
5,C6,C:\Users\sebam\AppData\Local\Temp\tmpardfzz93\...,2f733085a0fe
6,C7,C:\Users\sebam\AppData\Local\Temp\tmpardfzz93\...,16caee5ebfb3
7,C8,C:\Users\sebam\AppData\Local\Temp\tmpardfzz93\...,ec28f3789f43
8,C9,C:\Users\sebam\AppData\Local\Temp\tmpardfzz93\...,803537920bb2


Ok: Test 3 superado: fingerprint construido, recomputable y persistible.


## Test 4 - Determinismo exacto sobre una configuración representativa

Objetivo:
- comprobar que, con la misma seed y la misma configuración,
  el generador produce exactamente el mismo DataFrame,
  el mismo `TripSchema`,
  el mismo `schema_effective`,
  el mismo `manifest`,
  y el mismo `expected_fingerprint`.

Nota:
- este test usa `C2` por ser suficientemente representativa sin ser tan pesada como `C8` o `C9`.
- si quieres, luego puedes repetir el mismo patrón con otra configuración.

In [6]:
config = get_predefined_config("C2")

case_a = generate_experiment_case(config, seed=DEFAULT_SEED)
case_b = generate_experiment_case(config, seed=DEFAULT_SEED)

# 1) DataFrame exactamente igual
assert_frame_equal(
    case_a.trips.data,
    case_b.trips.data,
    check_dtype=True,
    check_exact=True,
    check_categorical=True,
)

# 2) TripSchema exactamente igual vía snapshot serializable
assert case_a.schema.to_dict() == case_b.schema.to_dict(), "TripSchema distinto entre llamadas repetidas"

# 3) schema_effective exactamente igual
assert case_a.trips.schema_effective.to_dict() == case_b.trips.schema_effective.to_dict(), (
    "TripSchemaEffective distinto entre llamadas repetidas"
)

# 4) manifest exactamente igual
assert case_a.manifest == case_b.manifest, "Manifest distinto entre llamadas repetidas"

# 5) fingerprint exactamente igual
assert case_a.expected_fingerprint == case_b.expected_fingerprint, (
    "Fingerprint esperado distinto entre llamadas repetidas"
)

_cleanup_case(case_a)
_cleanup_case(case_b)

_print_ok("Test 4 superado: determinismo exacto confirmado en C2.")

Ok: Test 4 superado: determinismo exacto confirmado en C2.


## Test 5 - Determinismo mínimo sobre las 9 configuraciones

Objetivo:
- comprobar en las 9 configuraciones que, con la misma seed, al menos se conservan:
  - `manifest`,
  - `TripSchema`,
  - `dataset_fingerprint`.

Este test es más liviano que comparar DataFrames completos por pares en `C8` y `C9`.

In [7]:
rows = []

for config_id in CONFIG_IDS:
    config = get_predefined_config(config_id)

    case_a = generate_experiment_case(config, seed=DEFAULT_SEED)
    case_b = generate_experiment_case(config, seed=DEFAULT_SEED)

    assert case_a.manifest == case_b.manifest, f"{config_id}: manifest no determinista"
    assert case_a.schema.to_dict() == case_b.schema.to_dict(), f"{config_id}: schema no determinista"
    assert (
        case_a.expected_fingerprint["dataset_fingerprint"]
        == case_b.expected_fingerprint["dataset_fingerprint"]
    ), f"{config_id}: dataset_fingerprint no determinista"

    rows.append(
        {
            "config_id": config_id,
            "dataset_fingerprint_prefix": case_a.expected_fingerprint["dataset_fingerprint"][:12],
        }
    )

    _cleanup_case(case_a)
    _cleanup_case(case_b)

det_df = pd.DataFrame(rows)
display(det_df)
_print_ok("Test 5 superado: determinismo mínimo confirmado en las 9 configuraciones.")

,config_id,dataset_fingerprint_prefix
0,C1,3f9af838fb90
1,C2,62c82cac6a35
2,C3,238689ae5886
3,C4,178d9db5a094
4,C5,fb3689520649
5,C6,2f733085a0fe
6,C7,16caee5ebfb3
7,C8,ec28f3789f43
8,C9,803537920bb2


Ok: Test 5 superado: determinismo mínimo confirmado en las 9 configuraciones.


## Test 6 - Validación explícita de que `get_predefined_config()` refleja exactamente las 9 configuraciones cerradas

Objetivo:
- comprobar que las 9 configuraciones del generador coinciden con las tuplas acordadas.

In [8]:
expected = {
    "C1": (36, 200_000, 10),
    "C2": (156, 200_000, 10),
    "C3": (256, 200_000, 10),
    "C4": (156, 20_000, 10),
    "C5": (156, 1_000_000, 10),
    "C6": (156, 200_000, 1_000),
    "C7": (156, 200_000, 10_000),
    "C8": (256, 1_000_000, 10),
    "C9": (156, 1_000_000, 10_000),
}

rows = []

for config_id, (exp_cols, exp_rows, exp_k) in expected.items():
    cfg = get_predefined_config(config_id)

    assert cfg.n_cols == exp_cols, f"{config_id}: n_cols incorrecto"
    assert cfg.n_rows == exp_rows, f"{config_id}: n_rows incorrecto"
    assert cfg.k == exp_k, f"{config_id}: k incorrecto"

    rows.append(
        {
            "config_id": config_id,
            "n_cols": cfg.n_cols,
            "n_rows": cfg.n_rows,
            "k": cfg.k,
        }
    )

expected_df = pd.DataFrame(rows)
display(expected_df)
_print_ok("Test 6 superado: las 9 configuraciones predefinidas coinciden con lo acordado.")

,config_id,n_cols,n_rows,k
0,C1,36,200000,10
1,C2,156,200000,10
2,C3,256,200000,10
3,C4,156,20000,10
4,C5,156,1000000,10
5,C6,156,200000,1000
6,C7,156,200000,10000
7,C8,256,1000000,10
8,C9,156,1000000,10000


Ok: Test 6 superado: las 9 configuraciones predefinidas coinciden con lo acordado.
